# 01 Load raw Municipal Money data into Lakehouse tables
Upload the six Parquet files from `data/raw/` into the Lakehouse under **Files/raw/** first, then run all cells.
Each file becomes a Delta table with the prefix `raw_`. No cleaning happens here: cleaning waits for the missingness decisions.

In [ ]:
import re
from notebookutils import mssparkutils

files = [f for f in mssparkutils.fs.ls("Files/raw") if f.name.endswith(".parquet")]
print([f.name for f in files])

In [ ]:
def clean_cols(df):
    # Delta column names can't contain dots or spaces: demarcation.code -> demarcation_code
    for c in df.columns:
        df = df.withColumnRenamed(c, re.sub(r"[^0-9a-zA-Z_]", "_", c))
    return df

for f in files:
    name = "raw_" + f.name.replace(".parquet", "")
    df = clean_cols(spark.read.parquet(f.path))
    df.write.mode("overwrite").format("delta").saveAsTable(name)
    print(f"{name}: {df.count():,} rows, {len(df.columns)} columns")

In [ ]:
%%sql
-- quick check: audit opinions per year
SELECT financial_year_end_year AS year, opinion_code, COUNT(*) AS n
FROM raw_audit_opinions
GROUP BY financial_year_end_year, opinion_code
ORDER BY year, opinion_code